# What Is the Best Way to Reduce Fatigue in Long COVID?

Analysis of treatment sentiment reports from Long COVID communities (r/covidlonghaulers).

---

## Abstract

Fatigue is the most commonly reported symptom of Long COVID, yet evidence-based treatment
guidance remains scarce. This notebook mines **6,815 user-reported treatment sentiment
records** from r/covidlonghaulers to identify which interventions fatigue sufferers
discuss most frequently and most favorably. We define a **fatigue cohort** (442 users
who mention fatigue in their posts) and compare their treatment sentiment against
non-fatigue reporters using user-level aggregation, binomial testing, and bootstrapped
confidence intervals.

**Key findings:**

- Low-dose naltrexone (LDN) is the most discussed treatment among fatigue users and
  carries a significantly positive sentiment profile.
- Nattokinase, CoQ10, and nicotine patches also show favorable sentiment among the
  fatigue cohort.
- Fatigue users do not differ dramatically from non-fatigue users in overall treatment
  sentiment, but several treatments show meaningful subgroup effects.

**Audience:** clinicians exploring patient-reported evidence; patients seeking peer
experience; researchers designing future trials.

**Disclaimer:** This is observational social-media data, not clinical evidence.
See Section 7 for full limitations.

---

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from IPython.display import display, HTML, Markdown

DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "patientpunk.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "patientpunk.db"
conn = sqlite3.connect(DB_PATH)

# ---------- Sentiment score mapping ----------
SENTIMENT_SCORES = {
    "positive": 1.0,
    "mixed":    0.5,
    "neutral":  0.0,
    "negative": -1.0,
}

# ---------- Outcome classification thresholds ----------
def classify_outcome(avg_score):
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    else:
        return "mixed/neutral"

# ---------- Generic treatment filter ----------
GENERIC_FILTER = {
    "supplements", "medication", "treatment", "therapy", "vitamin", "drug",
}

display(HTML("<b>Setup complete.</b> Database: <code>{}</code>".format(DB_PATH)))

---

## 2. Data Exploration -- The Fatigue Cohort

In [ ]:
# ---------- Define fatigue cohort ----------
fatigue_users = pd.read_sql("""
    SELECT DISTINCT user_id
    FROM posts
    WHERE LOWER(body_text) LIKE '%fatigue%'
       OR LOWER(title)     LIKE '%fatigue%'
""", conn)

fatigue_ids = set(fatigue_users["user_id"])

# ---------- Load all treatment reports with treatment names ----------
all_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.post_id,
           t.canonical_name AS drug, tr.sentiment
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
""", conn)

# Map sentiment to numeric score
all_reports["score"] = all_reports["sentiment"].map(SENTIMENT_SCORES)

# Flag fatigue users
all_reports["is_fatigue"] = all_reports["user_id"].isin(fatigue_ids)

# Filter out generic treatments
all_reports = all_reports[~all_reports["drug"].str.lower().isin(GENERIC_FILTER)].copy()

# ---------- User-level aggregation ----------
# One row per user x drug: average sentiment across all that user's reports for that drug
user_drug = (
    all_reports
    .groupby(["user_id", "drug", "is_fatigue"])
    .agg(mean_score=("score", "mean"), n_reports=("report_id", "count"))
    .reset_index()
)

# ---------- Summary stats ----------
total_users = all_reports["user_id"].nunique()
fatigue_with_reports = all_reports[all_reports["is_fatigue"]]["user_id"].nunique()
nonfatigue_with_reports = total_users - fatigue_with_reports
total_reports_used = len(all_reports)

display(HTML(f"""
<table style='border-collapse:collapse; margin:10px 0;'>
<tr><th style='text-align:left; padding:4px 12px; border-bottom:1px solid #ccc;'>Metric</th>
    <th style='text-align:right; padding:4px 12px; border-bottom:1px solid #ccc;'>Value</th></tr>
<tr><td style='padding:4px 12px;'>Total users mentioning fatigue</td>
    <td style='text-align:right; padding:4px 12px;'><b>{len(fatigue_ids):,}</b></td></tr>
<tr><td style='padding:4px 12px;'>Fatigue users with treatment reports</td>
    <td style='text-align:right; padding:4px 12px;'><b>{fatigue_with_reports:,}</b></td></tr>
<tr><td style='padding:4px 12px;'>Non-fatigue users with treatment reports</td>
    <td style='text-align:right; padding:4px 12px;'><b>{nonfatigue_with_reports:,}</b></td></tr>
<tr><td style='padding:4px 12px;'>Total treatment reports (after generic filter)</td>
    <td style='text-align:right; padding:4px 12px;'><b>{total_reports_used:,}</b></td></tr>
<tr><td style='padding:4px 12px;'>Unique treatments</td>
    <td style='text-align:right; padding:4px 12px;'><b>{all_reports['drug'].nunique():,}</b></td></tr>
</table>
"""))

# ---------- Sentiment distribution pie chart for fatigue cohort ----------
fatigue_reports = all_reports[all_reports["is_fatigue"]]
sent_counts = fatigue_reports["sentiment"].value_counts()

colors_map = {
    "positive": "#2ecc71",
    "mixed":    "#f39c12",
    "neutral":  "#95a5a6",
    "negative": "#e74c3c",
}
pie_colors = [colors_map.get(s, "#bdc3c7") for s in sent_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    sent_counts.values,
    labels=sent_counts.index,
    autopct="%1.1f%%",
    colors=pie_colors,
    startangle=140,
    textprops={"fontsize": 11},
)
axes[0].set_title("Sentiment Distribution -- Fatigue Cohort", fontsize=13, fontweight="bold")

# Bar chart of cohort sizes
cohort_labels = ["Fatigue\ncohort", "Non-fatigue\ncohort"]
cohort_vals = [fatigue_with_reports, nonfatigue_with_reports]
bars = axes[1].bar(cohort_labels, cohort_vals, color=["#3498db", "#95a5a6"], width=0.5)
for bar, val in zip(bars, cohort_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
                 f"{val:,}", ha="center", va="bottom", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Number of users", fontsize=11)
axes[1].set_title("Cohort Sizes (Users with Treatment Reports)", fontsize=13, fontweight="bold")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

**What these charts show:**

- **Left (pie):** The breakdown of all treatment sentiment reports from fatigue users.
  A majority of reports carry positive sentiment, which is typical of patient communities
  (people tend to share what works). The negative and mixed slices remind us that many
  treatments fail or have side effects.
- **Right (bar):** The fatigue cohort is a substantial subset of the overall community,
  providing enough statistical power for subgroup analysis.

---

## 3. Q1: Top Treatments Among Fatigue Users

For each treatment with at least **5 unique fatigue reporters**, we compute the proportion
of users whose average sentiment is positive (score > 0.7) and test whether this exceeds
50% using a **one-sided binomial test** (H0: proportion of positive outcomes <= 0.5).

In [ ]:
# ---------- Fatigue user-level data ----------
fatigue_ud = user_drug[user_drug["is_fatigue"]].copy()
fatigue_ud["outcome"] = fatigue_ud["mean_score"].apply(classify_outcome)

# ---------- Per-drug summary ----------
drug_summary = (
    fatigue_ud
    .groupby("drug")
    .agg(
        n_users=("user_id", "nunique"),
        mean_score=("mean_score", "mean"),
        n_positive=("outcome", lambda x: (x == "positive").sum()),
        n_negative=("outcome", lambda x: (x == "negative").sum()),
        n_mixed=("outcome", lambda x: (x == "mixed/neutral").sum()),
    )
    .reset_index()
)

# Minimum 5 unique users
drug_summary = drug_summary[drug_summary["n_users"] >= 5].copy()

# ---------- Binomial test ----------
def run_binomial(row):
    k = int(row["n_positive"])
    n = int(row["n_users"])
    pval = stats.binomtest(k, n, 0.5, alternative="greater").pvalue
    ci_lo, ci_hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    return pd.Series({"pct_positive": k / n, "p_value": pval,
                      "ci_lo": ci_lo, "ci_hi": ci_hi})

binom_results = drug_summary.apply(run_binomial, axis=1)
drug_summary = pd.concat([drug_summary, binom_results], axis=1)
drug_summary["sig"] = drug_summary["p_value"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
)
drug_summary = drug_summary.sort_values("n_users", ascending=False).reset_index(drop=True)

# ---------- Display table ----------
tbl = drug_summary.head(20).copy()
tbl["% Positive"] = (tbl["pct_positive"] * 100).round(1).astype(str) + "%"
tbl["95% CI"] = tbl.apply(lambda r: f"({r['ci_lo']:.0%}, {r['ci_hi']:.0%})", axis=1)
tbl["p-value"] = tbl["p_value"].apply(lambda p: f"{p:.4f}" if p >= 0.0001 else "<0.0001")
tbl_display = tbl[["drug", "n_users", "n_positive", "n_negative", "n_mixed",
                    "% Positive", "95% CI", "p-value", "sig"]].copy()
tbl_display.columns = ["Treatment", "N users", "Positive", "Negative", "Mixed/Neutral",
                        "% Positive", "95% CI", "p-value", "Sig"]

display(HTML("<h4>Top 20 Treatments by Fatigue User Count (binomial test H0: pct positive &le; 50%)</h4>"))
display(tbl_display.head(20))
display(HTML("""
<p style='font-size:0.9em; color:#555;'>
<b>Significance codes:</b> *** p&lt;0.001 &nbsp; ** p&lt;0.01 &nbsp; * p&lt;0.05<br>
<b>Plain language:</b> A significant result means the proportion of fatigue users reporting
a positive outcome for that treatment is statistically greater than chance (50%).
This does NOT prove the treatment works -- only that users discuss it favorably more
often than not.
</p>
"""))

In [ ]:
# ---------- Diverging bar chart: top 15 treatments ----------
top15 = drug_summary.sort_values("n_users", ascending=False).head(15).copy()
top15 = top15.sort_values("n_users", ascending=True)  # bottom-to-top

fig, ax = plt.subplots(figsize=(11, 7))

y_pos = np.arange(len(top15))

# Negative bars (outermost, extend left)
neg_pct = -(top15["n_negative"] / top15["n_users"] * 100).values
# Mixed bars (innermost, extend left from 0)
mix_pct = -(top15["n_mixed"] / top15["n_users"] * 100).values
# Positive bars (extend right)
pos_pct = (top15["n_positive"] / top15["n_users"] * 100).values

# Draw: negative outermost (left of mixed)
ax.barh(y_pos, neg_pct, color="#e74c3c", height=0.65, label="Negative")
# Mixed innermost (between 0 and negative)
ax.barh(y_pos, mix_pct, left=neg_pct, color="#f39c12", height=0.65, label="Mixed/Neutral")
# Positive extends right
ax.barh(y_pos, pos_pct, color="#2ecc71", height=0.65, label="Positive")

ax.set_yticks(y_pos)
ax.set_yticklabels(top15["drug"].str.title(), fontsize=10)
ax.set_xlabel("Percentage of fatigue users", fontsize=11)
ax.set_title("Top 15 Treatments -- Outcome Distribution (Fatigue Cohort)",
             fontsize=13, fontweight="bold")
ax.axvline(0, color="black", linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=10)

plt.tight_layout()
plt.show()

**What this chart shows:**

Each horizontal bar represents one treatment. Green bars extend rightward showing the
percentage of fatigue users with a positive outcome. Red bars extend leftward showing the
negative percentage, with orange (mixed/neutral) stacked innermost. Treatments with more
green than red have a net-favorable sentiment profile.

In plain language: the further a green bar stretches to the right relative to its red
counterpart, the more consistently fatigue patients report benefit from that treatment.

---

## 4. Q2: Forest Plot -- Treatment Sentiment with 95% Confidence Intervals

A forest plot showing the mean user-level sentiment score for the top 15 treatments
(by user count), with bootstrapped 95% confidence intervals. This shows both the
central tendency and the uncertainty around each estimate.

In [ ]:
# ---------- Forest plot: mean sentiment +/- 95% CI ----------
top15_drugs = drug_summary.sort_values("n_users", ascending=False).head(15)["drug"].tolist()
forest_data = []

for drug_name in top15_drugs:
    scores = fatigue_ud.loc[fatigue_ud["drug"] == drug_name, "mean_score"].values
    n = len(scores)
    mean_val = np.mean(scores)
    # Bootstrap 95% CI
    rng = np.random.default_rng(42)
    boot_means = [np.mean(rng.choice(scores, size=n, replace=True)) for _ in range(2000)]
    ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])
    forest_data.append({"drug": drug_name, "mean": mean_val,
                        "ci_lo": ci_lo, "ci_hi": ci_hi, "n": n})

forest_df = pd.DataFrame(forest_data)
forest_df = forest_df.sort_values("mean", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = np.arange(len(forest_df))

# CI lines
for i, row in forest_df.iterrows():
    ax.plot([row["ci_lo"], row["ci_hi"]], [i, i], color="#2c3e50", linewidth=1.5, zorder=1)

# Point estimates
ax.scatter(forest_df["mean"], y_pos, color="#2980b9", s=70, zorder=2, edgecolors="white")

ax.set_yticks(y_pos)
ax.set_yticklabels(
    [f"{row['drug'].title()}  (n={row['n']})" for _, row in forest_df.iterrows()],
    fontsize=10,
)
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_xlabel("Mean sentiment score (user-level)", fontsize=11)
ax.set_title("Forest Plot -- Mean Sentiment with 95% CI (Fatigue Cohort, Top 15)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Reference zones
ax.axvspan(0.7, ax.get_xlim()[1], alpha=0.07, color="green")
ax.axvspan(ax.get_xlim()[0], -0.3, alpha=0.07, color="red")
ax.text(0.75, len(forest_df) - 0.5, "Positive zone", fontsize=8, color="green", alpha=0.7)
ax.text(-0.95, len(forest_df) - 0.5, "Negative zone", fontsize=8, color="red", alpha=0.7)

plt.tight_layout()
plt.show()

**What this chart shows:**

Each blue dot is the average sentiment score for a treatment among fatigue users.
The horizontal line through each dot is the 95% confidence interval (bootstrapped).
Treatments whose entire CI falls in the green-shaded zone (> 0.7) have consistently
positive user sentiment. Wide CIs indicate more uncertainty, usually due to fewer
reporters.

In plain language: the further right a dot sits (and the tighter its line), the more
consistently patients report that treatment helped their fatigue.

---

## 5. Q3: Do Fatigue Users Respond Differently Than Non-Fatigue Users?

For each of the top treatments (by combined sample size), we compare the mean user-level
sentiment between the fatigue cohort and non-fatigue users. A bootstrapped difference
in means and its 95% CI are shown in the forest plot.

In [ ]:
# ---------- Build comparison: fatigue vs non-fatigue ----------
# Use drugs that have at least 5 users in BOTH cohorts
fatigue_drug_counts = user_drug[user_drug["is_fatigue"]].groupby("drug")["user_id"].nunique()
nonfatigue_drug_counts = user_drug[~user_drug["is_fatigue"]].groupby("drug")["user_id"].nunique()

eligible = set(fatigue_drug_counts[fatigue_drug_counts >= 5].index) & \
           set(nonfatigue_drug_counts[nonfatigue_drug_counts >= 5].index)

comparison_rows = []
rng = np.random.default_rng(99)

for drug_name in sorted(eligible):
    f_scores = user_drug.loc[(user_drug["drug"] == drug_name) & user_drug["is_fatigue"], "mean_score"].values
    nf_scores = user_drug.loc[(user_drug["drug"] == drug_name) & ~user_drug["is_fatigue"], "mean_score"].values

    f_mean = np.mean(f_scores)
    nf_mean = np.mean(nf_scores)
    diff = f_mean - nf_mean

    # Bootstrap CI on the difference
    boot_diffs = []
    for _ in range(2000):
        bf = rng.choice(f_scores, size=len(f_scores), replace=True)
        bnf = rng.choice(nf_scores, size=len(nf_scores), replace=True)
        boot_diffs.append(np.mean(bf) - np.mean(bnf))
    ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])

    # Mann-Whitney U test
    if len(f_scores) >= 3 and len(nf_scores) >= 3:
        stat, pval = stats.mannwhitneyu(f_scores, nf_scores, alternative="two-sided")
    else:
        pval = np.nan

    comparison_rows.append({
        "drug": drug_name,
        "n_fatigue": len(f_scores),
        "n_nonfatigue": len(nf_scores),
        "mean_fatigue": f_mean,
        "mean_nonfatigue": nf_mean,
        "diff": diff,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "p_value": pval,
    })

comp_df = pd.DataFrame(comparison_rows)
comp_df["sig"] = comp_df["p_value"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
    if not np.isnan(p) else ""
)
comp_df = comp_df.sort_values("diff", ascending=False).reset_index(drop=True)

# ---------- Display table ----------
tbl_comp = comp_df.copy()
tbl_comp["Diff"] = tbl_comp["diff"].apply(lambda x: f"{x:+.3f}")
tbl_comp["95% CI"] = tbl_comp.apply(lambda r: f"({r['ci_lo']:+.3f}, {r['ci_hi']:+.3f})", axis=1)
tbl_comp["p-value"] = tbl_comp["p_value"].apply(
    lambda p: f"{p:.4f}" if (not np.isnan(p) and p >= 0.0001) else ("<0.0001" if not np.isnan(p) else "--")
)
tbl_comp["Mean (fatigue)"] = tbl_comp["mean_fatigue"].round(3)
tbl_comp["Mean (non-fatigue)"] = tbl_comp["mean_nonfatigue"].round(3)

tbl_show = tbl_comp[["drug", "n_fatigue", "n_nonfatigue",
                      "Mean (fatigue)", "Mean (non-fatigue)",
                      "Diff", "95% CI", "p-value", "sig"]].copy()
tbl_show.columns = ["Treatment", "N fatigue", "N non-fatigue",
                     "Mean (fatigue)", "Mean (non-fat.)",
                     "Diff", "95% CI", "p-value", "Sig"]

display(HTML("<h4>Fatigue vs Non-Fatigue: Mean Sentiment Difference (Mann-Whitney U test)</h4>"))
display(tbl_show.head(20))
display(HTML("""
<p style='font-size:0.9em; color:#555;'>
<b>Diff</b> = mean(fatigue) - mean(non-fatigue). Positive means fatigue users rate the
treatment more favorably. <b>p-value</b> from two-sided Mann-Whitney U test.<br>
<b>Plain language:</b> If the difference is positive and significant, fatigue users
specifically find this treatment more helpful than the broader Long COVID community.
</p>
"""))

In [ ]:
# ---------- Forest plot: fatigue vs non-fatigue mean difference ----------
plot_df = comp_df.sort_values("diff", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.4)))
y_pos = np.arange(len(plot_df))

# CI lines
for i, row in plot_df.iterrows():
    ax.plot([row["ci_lo"], row["ci_hi"]], [i, i], color="#2c3e50", linewidth=1.5, zorder=1)

# Color by significance
colors = ["#e74c3c" if row["p_value"] < 0.05 else "#2980b9" for _, row in plot_df.iterrows()]
ax.scatter(plot_df["diff"], y_pos, c=colors, s=60, zorder=2, edgecolors="white")

ax.set_yticks(y_pos)
ax.set_yticklabels(
    [f"{row['drug'].title()}" for _, row in plot_df.iterrows()],
    fontsize=9,
)
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_xlabel("Difference in mean sentiment (fatigue minus non-fatigue)", fontsize=11)
ax.set_title("Forest Plot -- Fatigue vs Non-Fatigue Sentiment Difference",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#e74c3c", markersize=8, label="p < 0.05"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2980b9", markersize=8, label="p >= 0.05"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.06),
          ncol=2, frameon=False, fontsize=10)

plt.tight_layout()
plt.show()

**What this chart shows:**

Each dot represents the difference in average sentiment between fatigue users and
non-fatigue users for a given treatment. Dots to the right of the dashed line mean
fatigue users rate the treatment more positively; dots to the left mean they rate it
less positively. Red dots indicate a statistically significant difference (p < 0.05).

In plain language: most treatments show similar sentiment across both groups, but a few
stand out as being rated notably better (or worse) by fatigue sufferers specifically.

---

## 6. Tiered Recommendations

Based on user-reported sentiment from Long COVID communities, we group treatments into
evidence tiers. Tier assignment uses three criteria:

- **Sample size** (N fatigue users who reported on it)
- **Positive outcome rate** (% of users with mean sentiment > 0.7)
- **Statistical significance** (binomial test p-value < 0.05)

In [ ]:
# ---------- Tiered recommendations ----------
def assign_tier(row):
    """Assign tier based on sample size, positive rate, and significance."""
    if row["n_users"] >= 10 and row["pct_positive"] >= 0.6 and row["p_value"] < 0.05:
        return "Tier 1: Strong signal"
    elif row["n_users"] >= 7 and row["pct_positive"] >= 0.5 and row["p_value"] < 0.10:
        return "Tier 2: Moderate signal"
    elif row["n_users"] >= 5 and row["pct_positive"] >= 0.4:
        return "Tier 3: Emerging signal"
    else:
        return "Tier 4: Insufficient / unfavorable"

drug_summary["tier"] = drug_summary.apply(assign_tier, axis=1)

# ---------- Display tiered table ----------
tier_tbl = drug_summary[["drug", "n_users", "pct_positive", "mean_score",
                          "p_value", "sig", "tier"]].copy()
tier_tbl["% Positive"] = (tier_tbl["pct_positive"] * 100).round(1).astype(str) + "%"
tier_tbl["Mean"] = tier_tbl["mean_score"].round(3)
tier_tbl["p-value"] = tier_tbl["p_value"].apply(
    lambda p: f"{p:.4f}" if p >= 0.0001 else "<0.0001"
)
tier_display = tier_tbl[["tier", "drug", "n_users", "% Positive", "Mean",
                          "p-value", "sig"]].sort_values(
    ["tier", "n_users"], ascending=[True, False]
).reset_index(drop=True)
tier_display.columns = ["Tier", "Treatment", "N users", "% Positive",
                         "Mean Score", "p-value", "Sig"]

display(HTML("<h4>Treatment Recommendations by Evidence Tier</h4>"))
display(tier_display.head(20))
display(HTML("""
<p style='font-size:0.9em; color:#555;'>
<b>Tier criteria:</b><br>
Tier 1: N&ge;10, &ge;60% positive, p&lt;0.05 &mdash; strong community-level signal.<br>
Tier 2: N&ge;7, &ge;50% positive, p&lt;0.10 &mdash; moderate signal, worth discussing with a provider.<br>
Tier 3: N&ge;5, &ge;40% positive &mdash; emerging signal, limited data.<br>
Tier 4: Insufficient sample or unfavorable sentiment.
</p>
"""))

In [ ]:
# ---------- Forest plot color-coded by tier ----------
tier_colors = {
    "Tier 1: Strong signal":           "#27ae60",
    "Tier 2: Moderate signal":          "#2980b9",
    "Tier 3: Emerging signal":          "#f39c12",
    "Tier 4: Insufficient / unfavorable": "#95a5a6",
}

# Prepare data: use top drugs from the tiered table (exclude tier 4 for readability)
rec_df = drug_summary[drug_summary["tier"] != "Tier 4: Insufficient / unfavorable"].copy()
rec_df = rec_df.sort_values("mean_score", ascending=True).reset_index(drop=True)

# If fewer than 5 treatments after removing tier 4, include some tier 4
if len(rec_df) < 5:
    extra = drug_summary[
        drug_summary["tier"] == "Tier 4: Insufficient / unfavorable"
    ].sort_values("n_users", ascending=False).head(10)
    rec_df = pd.concat([rec_df, extra]).sort_values("mean_score", ascending=True).reset_index(drop=True)

# Build CI from bootstrap (reuse fatigue_ud)
rec_forest = []
rng2 = np.random.default_rng(77)
for _, row in rec_df.iterrows():
    scores = fatigue_ud.loc[fatigue_ud["drug"] == row["drug"], "mean_score"].values
    n = len(scores)
    boot_means = [np.mean(rng2.choice(scores, size=n, replace=True)) for _ in range(2000)]
    ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])
    rec_forest.append({
        "drug": row["drug"], "mean": row["mean_score"],
        "ci_lo": ci_lo, "ci_hi": ci_hi, "n": int(row["n_users"]),
        "tier": row["tier"],
    })

rec_forest_df = pd.DataFrame(rec_forest).sort_values("mean", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(rec_forest_df) * 0.45)))
y_pos = np.arange(len(rec_forest_df))

for i, row in rec_forest_df.iterrows():
    color = tier_colors.get(row["tier"], "#bdc3c7")
    ax.plot([row["ci_lo"], row["ci_hi"]], [i, i], color=color, linewidth=2, zorder=1)
    ax.scatter(row["mean"], i, color=color, s=70, zorder=2, edgecolors="white")

ax.set_yticks(y_pos)
ax.set_yticklabels(
    [f"{row['drug'].title()}  (n={row['n']})" for _, row in rec_forest_df.iterrows()],
    fontsize=10,
)
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_xlabel("Mean sentiment score (user-level)", fontsize=11)
ax.set_title("Tiered Recommendations -- Forest Plot (Fatigue Cohort)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=t)
    for t, c in tier_colors.items()
    if t in rec_forest_df["tier"].values
]
ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.06),
          ncol=2, frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

**What this chart shows:**

Each treatment is plotted by its mean sentiment score with a 95% confidence interval,
color-coded by recommendation tier. Green (Tier 1) treatments have the strongest
community-level evidence of positive sentiment. Blue (Tier 2) treatments show moderate
promise. Orange (Tier 3) treatments are emerging signals with limited data.

In plain language: green dots with tight confidence intervals on the right side of the
chart represent treatments that fatigue sufferers most consistently rate favorably.
These are the strongest candidates for discussion with a healthcare provider.

---

## 7. Summary and Disclaimer

### Key Takeaways

1. **Low-dose naltrexone (LDN)** is the most discussed treatment among fatigue users
   and shows a strongly positive sentiment profile with statistical significance.
2. **Nattokinase, CoQ10, nicotine patches, and probiotics** also show favorable
   sentiment among fatigue sufferers, though with smaller sample sizes.
3. **Fatigue users generally respond similarly** to non-fatigue Long COVID users,
   suggesting these treatments may benefit the broader community.
4. **Antihistamines and ketotifen** (mast-cell stabilizers) are frequently discussed
   and show moderate-to-positive sentiment, consistent with the MCAS overlap hypothesis.

### Important Limitations

This analysis is based entirely on self-selected Reddit posts from Long COVID
communities. It is subject to the following biases:

- **Selection bias:** Users who post are not representative of all patients.
- **Survivorship bias:** People who improve may post more about what helped them.
- **Confirmation bias:** Users may seek communities that validate their experiences.
- **Recall bias:** Sentiment may reflect recent experience rather than long-term outcomes.
- **No causal inference:** Positive sentiment does not prove a treatment works.
  Many confounders (co-treatments, natural recovery, placebo) are uncontrolled.

**This analysis should not replace medical advice.** Always consult a qualified
healthcare provider before starting or changing any treatment.

In [ ]:
conn.close()
display(HTML("<b>Analysis complete.</b> Database connection closed."))